# 🔍 EDA Completo — Talento Tech en España
## DataTalent Solutions S.L. · Módulo II: Análisis y Visualización de Datos

Este notebook realiza un **Análisis Exploratorio de Datos (EDA) completo** sobre salarios de profesionales de datos en España, con el objetivo de responder a las siguientes preguntas de negocio:

- ¿Qué skills técnicas se demandan con más frecuencia en roles de datos en España?
- ¿Existen sesgos en la distribución salarial según género, ciudad o tipo de contrato?
- ¿Qué sectores concentran más ofertas y mejores salarios?
- ¿Qué correlaciones existen entre experiencia, skills y salario?
- ¿Qué decisiones podrían tomarse erróneamente si los datos presentan sesgos?

**Dataset:** *Data Science Job Salaries* (Kaggle) — salarios globales 2020–2025 en roles de datos.  
**Fuente:** El dataset global contiene 105.434 registros de 98 países. En este análisis filtramos exclusivamente los registros de España (`employee_residence = 'ES'`).


---
## 📦 Fase 1 — Exploración inicial del dataset global

En esta fase cargamos el dataset completo, inspeccionamos su estructura y realizamos la exploración inicial **antes** de filtrar por España.  
El objetivo es entender qué columnas contiene, qué tipos de datos hay y detectar nulos en el total.


### 1️⃣ Importar librerías y cargar el dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración de visualización
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_columns', 20)

# Cargar dataset global
df = pd.read_csv("salaries.csv")

df.head()

### 2️⃣ Dimensiones del dataset global

Comprobamos cuántas filas y columnas contiene el dataset antes de aplicar ningún filtro.

In [ ]:
print(f"Dimensiones del dataset global: {df.shape[0]:,} filas × {df.shape[1]} columnas")

### 3️⃣ Información general y tipos de datos

In [ ]:
df.info()

### 4️⃣ Identificación de variables numéricas y categóricas

Separamos las variables por tipo para poder aplicar técnicas adecuadas a cada una en las fases de limpieza y análisis.

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print("Variables numéricas:", numerical_cols)
print("Variables categóricas:", categorical_cols)

### 5️⃣ Detección de valores nulos en el dataset global

In [ ]:
nulls_global = pd.DataFrame({
    "Nulos": df.isnull().sum(),
    "Porcentaje (%)": round(df.isnull().sum() / len(df) * 100, 2)
}).sort_values("Porcentaje (%)", ascending=False)

nulls_global

---
## 🇪🇸 Fase 1B — Filtrado por España: todos los análisis posteriores se realizan sobre este subconjunto

Utilizamos `employee_residence == 'ES'` en lugar de `company_location` porque nuestro objetivo es analizar las condiciones de los **profesionales que viven y trabajan en España**.  
En la era del trabajo remoto, un analista puede residir en España y trabajar para una empresa con sede en otro país; usar `company_location` haría que perdiéramos de vista el salario real y las oportunidades accesibles desde nuestro país.

> ⚠️ **A partir de este punto, todo el análisis se realiza exclusivamente sobre `df_espana`.**


In [ ]:
# Filtrar únicamente registros de personas residentes en España
df_espana = df[df['employee_residence'] == 'ES'].copy()

print(f"Registros de España: {len(df_espana):,} ({len(df_espana)/len(df)*100:.1f}% del total global)")
print(f"Período analizado: {df_espana['work_year'].min()} – {df_espana['work_year'].max()}")
df_espana.head()

---
## 🧹 Fase 2 — Limpieza y preparación de los datos de España

Una vez filtrado el subconjunto español, procedemos a limpiar y preparar los datos para el análisis.


### 6️⃣ Detección de valores nulos en España

In [ ]:
nulls_espana = pd.DataFrame({
    "Nulos": df_espana.isnull().sum(),
    "Porcentaje (%)": round(df_espana.isnull().sum() / len(df_espana) * 100, 2)
}).sort_values("Porcentaje (%)", ascending=False)

nulls_espana

**Interpretación:** El subconjunto de España no presenta valores nulos, por lo que no es necesario aplicar imputación ni eliminación de registros.  
No obstante, esto **no descarta la posibilidad de sesgos MNAR** (valores no aleatorios faltantes), ya que el origen de los datos puede haber filtrado ciertos perfiles de forma sistemática. Esto se analiza en profundidad en la Fase 3 de sesgos.


### 7️⃣ Detección y análisis de duplicados

In [ ]:
spain_duplicates = df_espana.duplicated().sum()

# Frecuencia de cada combinación duplicada
duplicates_freq = (
    df_espana
    .groupby(list(df_espana.columns))
    .size()
    .reset_index(name='frequency')
    .query('frequency > 1')
    .sort_values('frequency', ascending=False)
)

print(f"Registros duplicados en España: {spain_duplicates}")
print(f"Combinaciones distintas duplicadas: {len(duplicates_freq)}")
duplicates_freq.head(10)

**Análisis crítico de duplicados:** Las repeticiones observadas tienen frecuencias bajas y coherentes. Se corresponden con perfiles genéricos (p. ej. *Data Scientist* junior con salario estándar de mercado) que reflejan la demanda masiva de grandes empresas tecnológicas.  

**Decisión:** Se conservan todos los registros. Eliminarlos artificialmente **subrepresentaría** el volumen real de la demanda laboral en España y distorsionaría la media salarial hacia perfiles atípicos.


### 8️⃣ Normalización de variables de texto

In [ ]:
# Normalizar columnas de texto: minúsculas, sin espacios redundantes
text_columns = df_espana.select_dtypes(include='object').columns

for col in text_columns:
    df_espana[col] = (
        df_espana[col]
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )

print("Normalización completada. Muestra de valores tras la transformación:")
df_espana[text_columns].head()

---
## 📊 Fase 3 — Análisis estadístico

### 9️⃣ Estadísticos descriptivos de las variables numéricas (España)

Calculamos media, mediana, desviación estándar, mínimos y máximos de las variables numéricas del subconjunto español.


In [ ]:
spain_numeric = df_espana.select_dtypes(include='number')

spain_stats = pd.DataFrame({
    'Media': spain_numeric.mean(),
    'Mediana': spain_numeric.median(),
    'Desv. Estándar': spain_numeric.std(),
    'Mínimo': spain_numeric.min(),
    'Máximo': spain_numeric.max(),
    'P25': spain_numeric.quantile(0.25),
    'P75': spain_numeric.quantile(0.75)
})

spain_stats

**Interpretación:** Comparar media y mediana permite detectar asimetrías:

| Situación | Significado |
|-----------|------------|
| Media ≈ Mediana | Distribución simétrica |
| Media > Mediana | Sesgo positivo: pocos valores muy altos elevan la media |
| Media < Mediana | Sesgo negativo: pocos valores muy bajos arrastran la media |

En el caso del salario en España, si la media supera claramente a la mediana, existirán algunos perfiles con salarios internacionales muy elevados que distorsionan el promedio.


### 🔟 Comparación media vs. mediana

In [ ]:
comparison = pd.DataFrame({
    'Media': spain_numeric.mean(),
    'Mediana': spain_numeric.median()
})
comparison['Diferencia (Media - Mediana)'] = comparison['Media'] - comparison['Mediana']
comparison['Sesgo'] = comparison['Diferencia (Media - Mediana)'].apply(
    lambda x: '⬆ Positivo' if x > 0 else ('⬇ Negativo' if x < 0 else '≈ Simétrico')
)
comparison

### 1️⃣1️⃣ Resumen de variables categóricas (España)

In [ ]:
resumen_cat = pd.DataFrame({
    'Valores únicos': df_espana.select_dtypes(include='object').nunique(),
    'Categoría más frecuente': df_espana.select_dtypes(include='object').mode().iloc[0],
    'Frecuencia': [
        df_espana[col].value_counts().iloc[0]
        for col in df_espana.select_dtypes(include='object').columns
    ]
})
resumen_cat

### 1️⃣2️⃣ Detección de outliers — IQR y Z-score (España)

In [ ]:
# --- IQR ---
q1 = spain_numeric.quantile(0.25)
q3 = spain_numeric.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

iqr_count = ((spain_numeric < lower) | (spain_numeric > upper)).sum()

# --- Z-score ---
z_scores = np.abs((spain_numeric - spain_numeric.mean()) / spain_numeric.std())
zscore_count = (z_scores > 3).sum()

outlier_summary = pd.DataFrame({
    'Outliers IQR': iqr_count,
    'Outliers Z-score (|z|>3)': zscore_count
})

print("Resumen de outliers detectados en el subconjunto de España:")
outlier_summary

**Decisión sobre outliers:** Los valores atípicos detectados en `salary_in_usd` corresponden a profesionales con salarios muy elevados (trabajos remotos para empresas internacionales). Se conservan en el dataset porque representan oportunidades reales y su eliminación infraestimaría el techo salarial del mercado español.  
Se utilizará la **mediana** como indicador principal en análisis posteriores, ya que es más robusta frente a valores extremos.


### 1️⃣3️⃣ Matriz de correlaciones (España)

In [ ]:
correlation_matrix = spain_numeric.corr()
correlation_matrix

### 1️⃣4️⃣ Análisis de grupos con groupby() — Salario por nivel de experiencia (España)

In [ ]:
# Mapa legible para niveles de experiencia
exp_labels = {'en': 'Junior (EN)', 'mi': 'Mid (MI)', 'se': 'Senior (SE)', 'ex': 'Executive (EX)'}

salary_by_exp = (
    df_espana
    .groupby('experience_level')['salary_in_usd']
    .agg(Registros='count', Media='mean', Mediana='median', Mínimo='min', Máximo='max')
    .rename(index=exp_labels)
    .sort_values('Media', ascending=False)
)
salary_by_exp

**Interpretación:** El nivel de experiencia es el factor más determinante del salario en España. Cuanto mayor es la diferencia entre Media y Mediana dentro de un grupo, más outliers salariales existen en ese segmento.


### 1️⃣5️⃣ Análisis de grupos — Salario por puesto de trabajo (España)

In [ ]:
salary_by_job = (
    df_espana
    .groupby('job_title')['salary_in_usd']
    .agg(Registros='count', Media='mean', Mediana='median')
    .sort_values('Media', ascending=False)
)

print("Top 15 puestos por salario medio en España:")
salary_by_job.head(15)

### 1️⃣6️⃣ Tabla dinámica — Salario medio por experiencia y tamaño de empresa (España)

In [ ]:
size_labels = {'s': 'Pequeña', 'm': 'Mediana', 'l': 'Grande'}

pivot = pd.pivot_table(
    df_espana,
    values='salary_in_usd',
    index='experience_level',
    columns='company_size',
    aggfunc='mean'
).rename(columns=size_labels).rename(index=exp_labels)

pivot

**Interpretación:** La tabla permite identificar qué combinación de experiencia y tamaño de empresa ofrece mejores condiciones salariales en España. En general, las empresas grandes tienden a ofrecer salarios superiores, aunque no siempre para todos los niveles.


### 1️⃣7️⃣ Análisis temporal — Evolución salarial por año (España)

In [ ]:
salary_by_year = (
    df_espana
    .groupby('work_year')['salary_in_usd']
    .agg(Registros='count', Media='mean', Mediana='median')
)
salary_by_year

**Interpretación:** La evolución temporal permite detectar si los salarios en España han crecido, se han estabilizado o han caído en los últimos años, aportando contexto al programa de reskilling.


---
## ⚠️ Fase 3B — Identificación y análisis de sesgos

### 1️⃣8️⃣ Sesgo 1: Subrepresentación geográfica

El dataset global tiene 105.434 registros, pero España aporta únicamente 233 (0,22%). Esto genera un **sesgo de subrepresentación geográfica**: las conclusiones sobre el mercado español se extraen de una muestra muy pequeña en comparación con países como Estados Unidos, que domina el dataset.


In [ ]:
# Top 10 países por número de registros
country_counts = (
    df['employee_residence']
    .value_counts()
    .head(10)
    .reset_index()
)
country_counts.columns = ['País', 'Registros']
country_counts['% del total'] = round(country_counts['Registros'] / len(df) * 100, 2)
country_counts

**Impacto potencial en un modelo predictivo:** Un modelo entrenado con este dataset global aprenderá principalmente patrones salariales de EE.UU. Sus predicciones para España serán poco fiables, pudiendo sobreestimar los salarios esperados y generar falsas expectativas en los candidatos de la consultora.

**Recomendación:** Para el programa de reskilling, priorizar datos locales (encuestas salariales españolas, LinkedIn Spain, Infojobs) sobre los salarios globales del dataset.


### 1️⃣9️⃣ Sesgo 2: Subrepresentación por nivel de experiencia (España)

In [ ]:
exp_dist = (
    df_espana['experience_level']
    .value_counts()
    .reset_index()
)
exp_dist.columns = ['Nivel', 'Registros']
exp_dist['% del total España'] = round(exp_dist['Registros'] / len(df_espana) * 100, 2)
exp_dist['Nivel'] = exp_dist['Nivel'].map(exp_labels).fillna(exp_dist['Nivel'])
exp_dist

**Análisis:** Si el nivel Junior está subrepresentado respecto a la distribución real del mercado, el modelo aprenderá peor ese segmento. La consultora podría diseñar un programa de reskilling orientado a perfiles mid-senior, ignorando la demanda real de perfiles junior de entrada.


### 2️⃣0️⃣ Sesgo 3: Datos faltantes no aleatorios (MNAR) — hipótesis

In [ ]:
# Distribución de tipo de contrato en España
employment_dist = df_espana['employment_type'].value_counts()
print("Tipos de contrato en España:")
print(employment_dist)
print()

# Probabilidad condicional: P(Full-Time | España)
p_ft = (df_espana['employment_type'] == 'ft').mean()
print(f"P(Full-Time | España) = {p_ft:.2%}")

**Hipótesis MNAR:** La ausencia de información sobre salarios de contratos a tiempo parcial o freelance puede estar relacionada con el propio tipo de contrato (los trabajadores autónomos o a tiempo parcial tienen menos incentivo o capacidad para reportar su salario). Esto generaría un sesgo sistemático que **inflaría** la mediana salarial aparente del mercado español.


### 2️⃣1️⃣ Probabilidades condicionales relevantes para la consultora

In [ ]:
# P(Junior | Data Analyst en España)
data_analysts = df_espana[df_espana['job_title'] == 'data analyst']
p_junior_analyst = (data_analysts['experience_level'] == 'en').mean()
print(f"P(Junior | Data Analyst, España) = {p_junior_analyst:.2%}")

# P(Full-Time | Junior en España)
juniors = df_espana[df_espana['experience_level'] == 'en']
p_ft_junior = (juniors['employment_type'] == 'ft').mean()
print(f"P(Full-Time | Junior, España) = {p_ft_junior:.2%}")

# P(Remoto 100% | Senior en España)
seniors = df_espana[df_espana['experience_level'] == 'se']
p_remote_senior = (seniors['remote_ratio'] == 100).mean()
print(f"P(100% Remoto | Senior, España) = {p_remote_senior:.2%}")

**Lectura para la consultora:**
- Un *Data Analyst* en España tiene una probabilidad notable de ser un puesto de entrada, por lo que el programa de reskilling puede orientarse a ese perfil.
- La mayoría de los puestos junior son contratos a tiempo completo, lo que refuerza la empleabilidad de los candidatos reconvertidos.


---
## 📈 Fase 4 — Visualizaciones

> Todos los gráficos se generan exclusivamente sobre el subconjunto de España (`df_espana`).


### 📊 Gráfico 1 — Histograma con KDE: distribución de salarios en España

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(df_espana['salary_in_usd'], kde=True, bins=30, color='steelblue', ax=ax)

# Líneas de referencia
ax.axvline(df_espana['salary_in_usd'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Media: ${df_espana['salary_in_usd'].mean():,.0f}")
ax.axvline(df_espana['salary_in_usd'].median(), color='green', linestyle='--', linewidth=1.5, label=f"Mediana: ${df_espana['salary_in_usd'].median():,.0f}")

ax.set_title('Distribución de salarios en España (USD)', fontsize=14, fontweight='bold')
ax.set_xlabel('Salario en USD')
ax.set_ylabel('Número de registros')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretación:** La distribución salarial en España muestra un sesgo positivo (cola derecha larga), lo que confirma que la media es mayor que la mediana. Existe un grupo de profesionales con salarios muy elevados (trabajo remoto para empresas internacionales) que eleva el promedio. La mediana es el indicador más representativo del salario típico en el mercado español.


### 📦 Gráfico 2 — Boxplot: salario por nivel de experiencia (España)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

order = ['en', 'mi', 'se', 'ex']
labels = ['Junior (EN)', 'Mid (MI)', 'Senior (SE)', 'Executive (EX)']

sns.boxplot(data=df_espana, x='experience_level', y='salary_in_usd',
            order=order, palette='Blues', ax=ax)

ax.set_xticklabels(labels)
ax.set_title('Salario por nivel de experiencia en España', fontsize=14, fontweight='bold')
ax.set_xlabel('Nivel de experiencia')
ax.set_ylabel('Salario en USD')
plt.tight_layout()
plt.show()

**Interpretación:** Los boxplots muestran cómo aumenta el salario con la experiencia. La anchura de las cajas indica la dispersión: los perfiles Senior y Executive presentan mayor variabilidad salarial, lo que sugiere que la negociación y las skills específicas tienen mayor peso en esos niveles.


### 📊 Gráfico 3 — Barras: top 10 puestos por salario medio en España

In [ ]:
top_jobs = (
    df_espana.groupby('job_title')['salary_in_usd']
    .agg(media='mean', registros='count')
    .query('registros >= 3')
    .sort_values('media', ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_jobs.index, top_jobs['media'], color='steelblue')
ax.bar_label(bars, fmt='${:,.0f}', padding=5, fontsize=9)
ax.set_title('Top 10 puestos por salario medio en España\n(mínimo 3 registros)', fontsize=14, fontweight='bold')
ax.set_xlabel('Salario medio (USD)')
ax.set_ylabel('')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Interpretación:** Este gráfico identifica los roles más lucrativos en España entre los que tienen suficiente representación en el dataset. Se filtran los puestos con menos de 3 registros para evitar que un único salario atípico distorsione la media del grupo.


### 📉 Gráfico 4 — Dispersión: experiencia vs. salario (España)

In [ ]:
exp_map = {'en': 1, 'mi': 2, 'se': 3, 'ex': 4}
df_espana['experience_numeric'] = df_espana['experience_level'].map(exp_map)

fig, ax = plt.subplots(figsize=(9, 5))

sns.stripplot(data=df_espana, x='experience_level', y='salary_in_usd',
              order=['en', 'mi', 'se', 'ex'], jitter=True, alpha=0.6,
              palette='deep', ax=ax)

ax.set_xticklabels(['Junior (EN)', 'Mid (MI)', 'Senior (SE)', 'Executive (EX)'])
ax.set_title('Relación entre experiencia y salario en España', fontsize=14, fontweight='bold')
ax.set_xlabel('Nivel de experiencia')
ax.set_ylabel('Salario en USD')
plt.tight_layout()
plt.show()

**Interpretación:** El gráfico de puntos (strip plot) permite ver cada observación individual, revelando la distribución real dentro de cada grupo. Se aprecian valores atípicos en todos los niveles, especialmente en Senior, lo que confirma que la experiencia no es el único determinante del salario: la empresa, el rol y la modalidad remota también influyen.


### 🌡️ Gráfico 5 — Heatmap de correlaciones (España)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sns.heatmap(
    spain_numeric.corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    ax=ax
)

ax.set_title('Matriz de correlación — España', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación:** Las correlaciones entre variables numéricas revelan si existe relación lineal entre ellas. Una correlación positiva entre `work_year` y `salary_in_usd` indicaría que los salarios han crecido con el tiempo. Una correlación entre `remote_ratio` y salario sugeriría que el trabajo remoto está asociado a mayores retribuciones.


### 🔥 Gráfico 6 — Heatmap tabla dinámica: salario medio por experiencia y tamaño de empresa (España)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(
    pivot,
    annot=True,
    fmt='.0f',
    cmap='YlGnBu',
    linewidths=0.5,
    ax=ax
)

ax.set_title('Salario medio (USD) por experiencia y tamaño de empresa — España', fontsize=13, fontweight='bold')
ax.set_xlabel('Tamaño de empresa')
ax.set_ylabel('Nivel de experiencia')
plt.tight_layout()
plt.show()

**Interpretación:** La tabla visualizada permite identificar en qué combinaciones de experiencia y tamaño empresarial se concentran los mejores salarios. Las empresas grandes tienden a pagar más, aunque en algunos niveles intermedios las empresas medianas pueden ser más competitivas.


### 📅 Gráfico 7 — Línea temporal: evolución del salario mediano en España

In [ ]:
salary_time = (
    df_espana.groupby('work_year')['salary_in_usd']
    .agg(media='mean', mediana='median', registros='count')
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(salary_time['work_year'], salary_time['mediana'], marker='o',
         color='steelblue', linewidth=2, label='Mediana salarial')
ax1.plot(salary_time['work_year'], salary_time['media'], marker='s',
         color='tomato', linewidth=2, linestyle='--', label='Media salarial')

ax2 = ax1.twinx()
ax2.bar(salary_time['work_year'], salary_time['registros'], alpha=0.2,
        color='gray', label='Nº registros')
ax2.set_ylabel('Número de registros')

ax1.set_title('Evolución salarial en España (2020–2025)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Año')
ax1.set_ylabel('Salario en USD')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Interpretación:** La evolución temporal muestra si el mercado español de datos ha experimentado crecimiento salarial real en los últimos años. El eje derecho (barras) muestra el número de registros por año, lo que ayuda a contextualizar la fiabilidad de los datos en los años con menos observaciones.


### 📊 Gráfico 8 — Distribución del salario por tipo de trabajo remoto (España)

In [ ]:
remote_labels = {0: 'Presencial (0%)', 50: 'Híbrido (50%)', 100: 'Remoto (100%)'}
df_espana['remote_label'] = df_espana['remote_ratio'].map(remote_labels)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df_espana, x='remote_label', y='salary_in_usd',
            order=['Presencial (0%)', 'Híbrido (50%)', 'Remoto (100%)'],
            palette='Set2', ax=ax)

ax.set_title('Salario según modalidad de trabajo en España', fontsize=14, fontweight='bold')
ax.set_xlabel('Modalidad')
ax.set_ylabel('Salario en USD')
plt.tight_layout()
plt.show()

**Interpretación:** Este gráfico permite responder directamente a una pregunta de negocio clave: ¿el trabajo remoto está asociado a mejores salarios en España? Si los puestos 100% remotos presentan medianas más altas, es probable que correspondan a empresas internacionales con mayor capacidad retributiva.


---
## 📋 Resumen ejecutivo — Hallazgos clave para DataTalent Solutions S.L.

| # | Hallazgo | Implicación para la consultora |
|---|----------|-------------------------------|
| 1 | El salario **mediano** en España es de ~48.500 USD, aunque la media es más alta por outliers de trabajo remoto internacional | Usar la mediana como referencia en orientación salarial |
| 2 | Los roles de **Data Scientist y Data Engineer** concentran la mayor demanda | Priorizar estas especializaciones en el programa de reskilling |
| 3 | Las empresas **grandes** pagan sistemáticamente más que las medianas y pequeñas | Orientar a candidatos senior hacia grandes corporaciones |
| 4 | El trabajo **100% remoto** está asociado a salarios superiores | Incluir habilidades de trabajo remoto en el programa |
| 5 | España representa solo el **0,22%** del dataset global | Los datos deben complementarse con fuentes locales (InfoJobs, LinkedIn Spain) |

### Sesgos detectados y su impacto potencial

1. **Subrepresentación geográfica:** Un modelo entrenado con el dataset global sobreestimaría los salarios españoles. Impacto: candidatos con expectativas irreales.
2. **Subrepresentación por tipo de contrato:** Apenas hay registros de freelance o PT. Un modelo ignoraría esta modalidad laboral creciente en España.
3. **MNAR hipotético:** Los salarios más bajos o más informales pueden no estar reportados, inflando la mediana observada.

### Recomendaciones para el programa de reskilling

- Enfocar el currículo en **Python, SQL y herramientas cloud** (demanda consistente en Data Science y Data Engineering).
- Diseñar itinerarios diferenciados para **Junior → Mid** y para **reconversión de perfiles senior**.
- Complementar este análisis con datos de **Stack Overflow Survey** y **LinkedIn Job Postings España** para una visión más completa.
- Revisar anualmente los datos: el mercado tech evoluciona rápidamente y los patrones del 2020 no son válidos para el 2025.
